# Week 8 Capstone — Meeting-to-Action Assistant

Self-contained integration of **Week 3 structured output**, **Week 4 validation**, **Week 5 MiniLM+FAISS RAG**, **Week 6 cross-encoder reranking**, and **Week 7 dynamic few-shot selection**.

**A/B/C:** A transcript only; B + retrieved/reranked operational guidance; C + dynamic few-shot examples. Gemini produces the brief; a small local Qwen model is measured separately on narrow classification.


In [ ]:
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy tabulate transformers torch
import os,re,json,time,math,random,hashlib,numpy as np,pandas as pd,faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown
try:
 from google.colab import userdata
 if not os.getenv("GEMINI_API_KEY"): os.environ["GEMINI_API_KEY"]=userdata.get("GEMINI_API_KEY")
except: pass
GM=os.getenv("GEMINI_MODEL","gemini-3.5-flash");JM=os.getenv("GEMINI_JUDGE_MODEL",GM);QM="Qwen/Qwen2.5-0.5B-Instruct"
CACHE={"s":{},"j":{}};N={"s":0,"j":0}
def key(x):return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()
def cli():
 from openai import OpenAI
 return OpenAI(api_key=os.environ["GEMINI_API_KEY"],base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
def ask(m,p):
 for i,w in enumerate([2,4,8,16,32,32]):
  try:return cli().chat.completions.create(model=m,messages=[{"role":"user","content":p}],temperature=0)
  except Exception as e:
   if getattr(e,"status_code",None) not in {429,500,502,503,504} or i==5:raise
   time.sleep(w+random.random())
def J(s):return json.loads(re.sub(r"^```(?:json)?\s*|\s*```$","",s.strip(),flags=re.I))
print(GM,JM,QM)


## Held-out cases and reused retrieval components


In [ ]:
C=[
("E01","Maybe move Monday. No, keep Friday. I'll send QA.",["keep friday"],[("send qa","B")],[],[]),
("E02","Export missing IDs. I'll ask Finance.",[],[("ask finance","B")],["missing ids"],[]),
("E03","Could we use vendor B? Legal has not reviewed it. Do not choose yet.",["do not choose"],[],["legal has not reviewed"],[]),
("E04","Weekly dashboard might help. No cadence decision. I'll collect examples.",[],[("collect examples","B")],[],[]),
("E05","Import failed on invalid dates. I'll fix dates. Keep current mapping.",["keep current mapping"],[("fix dates","B")],["invalid dates"],[]),
("E06","Use Sep 1-Sep 30 for both reports. I'll rerun them.",["use sep 1 sep 30"],[("rerun","B")],[],[]),
("E07","We could automate email. Not in v1; human review required. I'll document the gate.",["human review required"],[("document gate","A")],[],[]),
("E08","Duplicate customers cause bad totals. I'll test customer ID.",[],[("test customer id","B")],["duplicate customers"],[]),
("E09","Should we remove notes? Not yet. Keep the shorter form. I'll measure time.",["keep shorter form"],[("measure time","A")],[],["remove notes"]),
("E10","SSO effort unknown. Do not promise it. I'll ask Engineering. Need separate coreference model?",["do not promise"],[("ask engineering","B")],["sso effort unknown"],["coreference model"])]
R=["Decisions require explicit commitment.","Maybe/could/might suggestions are not decisions unless later committed.","Actions are future work; never invent an owner.","Blockers prevent, delay, or risk progress.","Questions must remain unresolved.","Extracted items need transcript evidence."]
S=[("Maybe delay. No, keep Friday.","decision"),("Could we use vendor B? Compare terms first.","no decision"),("Missing IDs. I'll ask Finance.","action"),("Legal not reviewed.","blocker"),("Should we remove notes?","question")]
display(pd.DataFrame([{"case":x[0],"transcript":x[1]} for x in C]));display(pd.DataFrame({"RAG guidance":R}));display(pd.DataFrame(S,columns=["few-shot input","target"]))
E=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2");X=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def idx(d):v=E.encode(d,normalize_embeddings=True).astype("float32");z=faiss.IndexFlatIP(v.shape[1]);z.add(v);return z
RI,SI=idx(R),idx([x[0] for x in S])
def ret(q,d,z,txt=lambda x:x,k=3):
 v=E.encode([q],normalize_embeddings=True).astype("float32");a,b=z.search(v,min(6,len(d)));c=[d[int(i)] for i in b[0] if i>=0];s=X.predict([(q,txt(x)) for x in c]);return [x for _,x in sorted(zip(s,c),reverse=True)[:k]]
display(pd.DataFrame({"sample retrieved rules":ret(C[0][1],R,RI,k=4)}))


## A/B/C evaluation, judge, failure, and mitigation


In [ ]:
P='''Return JSON only: {"executive_summary":"string","decisions":[{"decision":"string","evidence":"string"}],"action_items":[{"description":"string","owner":"string","evidence":"string"}],"blockers":[{"blocker":"string","evidence":"string"}],"follow_up_questions":[{"question":"string","evidence":"string"}]}. Only confirmed decisions; do not invent owners; blockers must delay/risk progress; unresolved questions only; include evidence.'''
REQ=["executive_summary","decisions","action_items","blockers","follow_up_questions"]
def ex(t,g):
 k=key([GM,g,t])
 if k in CACHE["s"]:return CACHE["s"][k]
 p=[P];log={"rules":[],"examples":[]}
 if g in "BC":log["rules"]=ret(t,R,RI,k=4);p+=["Guidance: "+str(log["rules"])]
 if g=="C":log["examples"]=ret(t,S,SI,lambda x:x[0],3);p+=["Examples: "+str(log["examples"])]
 st=time.time()
 try:b=J(ask(GM,chr(10).join(p+["Transcript:",t])).choices[0].message.content)
 except:b={}
 o={"brief":b,"valid":isinstance(b,dict) and all(x in b for x in REQ),"lat":time.time()-st,"log":log};CACHE["s"][k]=o;return o
def toks(x):return set(re.findall(r"[a-z0-9]+",str(x).lower()))
def f1(p,g):
 u=set();tp=0
 for x in p:
  z=[(len(toks(x)&toks(y))/max(1,len(toks(x)|toks(y))),i) for i,y in enumerate(g) if i not in u]
  if z and max(z)[0]>=.25:tp+=1;u.add(max(z)[1])
 P=tp/len(p) if p else (1 if not g else 0);Q=tp/len(g) if g else (1 if not p else 0);return 2*P*Q/(P+Q) if P+Q else 0
def sc(c,r):
 b=r["brief"] if r["valid"] else {};d=[x.get("decision","") for x in b.get("decisions",[])];a=[x.get("description","") for x in b.get("action_items",[])];o=[x.get("owner","") for x in b.get("action_items",[])];bl=[x.get("blocker","") for x in b.get("blockers",[])];q=[x.get("question","") for x in b.get("follow_up_questions",[])];own=1 if not c[3] else int(any(i<len(o) and o[i].lower()==go.lower() for i in range(len(o)) for _,go in c[3]));return {"case":c[0],"decision_f1":f1(d,c[2]),"action_f1":f1(a,[x[0] for x in c[3]]),"owner_accuracy":own,"blocker_f1":f1(bl,c[4]),"question_f1":f1(q,c[5]),"valid":int(r["valid"]),"latency_s":r["lat"]}
RES={};rows=[]
if os.getenv("GEMINI_API_KEY"):
 for g in "ABC":
  RES[g]={}
  for c in C:r=ex(c[1],g);RES[g][c[0]]=r;x=sc(c,r);x["config"]=g;rows.append(x)
 M=pd.DataFrame(rows);SUM=M.groupby("config").mean(numeric_only=True).round(3);SUM["score"]=SUM[["decision_f1","action_f1","owner_accuracy","blocker_f1","question_f1","valid"]].mean(axis=1).round(3);BEST=SUM.score.idxmax();display(M);display(SUM);print("Selected",BEST)
else:print("API-BLOCKED")
JD=pd.DataFrame();JS=pd.DataFrame()
if RES:
 z=[]
 for c in C:
  k=key([JM,c[0],RES[BEST][c[0]]["brief"]]);payload={"transcript":c[1],"candidate":RES[BEST][c[0]]["brief"],"gold":c[2:]}
  try:
   j=CACHE["j"].get(k) or J(ask(JM,'Score 1-5 on correctness, grounding, operational_usefulness. Return JSON only with those keys plus "reason".'+chr(10)+json.dumps(payload)).choices[0].message.content);CACHE["j"][k]=j;z.append({"case":c[0],**j})
  except Exception as e:print("judge",c[0],e)
 JD=pd.DataFrame(z)
 if not JD.empty:JS=JD[["correctness","grounding","operational_usefulness"]].mean().to_frame("average").round(3);display(JD);display(JS)
FAIL=None;BEF=AFT=None
if RES:
 s={g:{c[0]:sc(c,RES[g][c[0]]) for c in C} for g in "ABC"};ch=co=None
 for c in C:
  if s["C"][c[0]]["decision_f1"]<s["B"][c[0]]["decision_f1"]:ch,co=c[0],"dynamic few-shot";break
 if ch is None:
  for c in C:
   if s["B"][c[0]]["decision_f1"]<s["A"][c[0]]["decision_f1"]:ch,co=c[0],"retrieval/reranking";break
 if ch is None:ch=min((c[0] for c in C),key=lambda x:s[BEST][x]["decision_f1"]);co="generation"
 FAIL={"case":ch,"component":co};print(FAIL)
 def mit(b):
  z=json.loads(json.dumps(b));H=["maybe","could","might"];K=["agreed","keep","use ","do not","required","will"];z["decisions"]=[d for d in z.get("decisions",[]) if not(any(x in d.get("evidence","").lower() for x in H) and not any(x in d.get("evidence","").lower() for x in K))];return z
 x=[];y=[]
 for c in C:r=RES[BEST][c[0]];x.append(sc(c,r)["decision_f1"]);r2=dict(r);r2["brief"]=mit(r["brief"]) if r["valid"] else {};r2["valid"]=all(k in r2["brief"] for k in REQ);y.append(sc(c,r2)["decision_f1"])
 BEF,AFT=round(float(np.mean(x)),3),round(float(np.mean(y)),3);print("Mitigation",BEF,"->",AFT)


## Gemini vs. small local Qwen


In [ ]:
T=[("Maybe move launch.","context"),("Keep Friday.","decision"),("I'll send QA.","action_item"),("SSO unknown.","blocker"),("What is SSO effort?","follow_up_question"),("No cadence decision.","context"),("Keep mapping.","decision"),("I'll fix dates.","action_item"),("Legal not reviewed.","blocker"),("Which agreement can Legal clear?","follow_up_question")]
GDF=QDF=CMP=pd.DataFrame()
if os.getenv("GEMINI_API_KEY"):
 try:
  p="Classify each span as decision, action_item, blocker, follow_up_question, context. Return JSON only as {\"labels\":[]}."+chr(10)+chr(10).join(f"{i+1}. {t}" for i,(t,_) in enumerate(T));st=time.time();o=J(ask(GM,p).choices[0].message.content);lat=(time.time()-st)/len(T);GDF=pd.DataFrame([{"gold":g,"pred":q,"correct":g==q,"lat":lat} for (_,g),q in zip(T,o["labels"])]);display(GDF)
 except Exception as e:print(e)
try:
 import torch
 from transformers import AutoTokenizer,AutoModelForCausalLM
 tok=AutoTokenizer.from_pretrained(QM);m=AutoModelForCausalLM.from_pretrained(QM,torch_dtype="auto");m.eval()
 def qc(t):
  x=tok(tok.apply_chat_template([{"role":"user","content":"Return one label only: decision, action_item, blocker, follow_up_question, context. "+t}],tokenize=False,add_generation_prompt=True),return_tensors="pt");st=time.time()
  with torch.no_grad():y=m.generate(**x,max_new_tokens=8,do_sample=False)
  return re.sub(r"[^a-z_]","",tok.decode(y[0][x["input_ids"].shape[1]:],skip_special_tokens=True).strip().lower()),time.time()-st
 QDF=pd.DataFrame([{"gold":g,"pred":p,"correct":g==p,"lat":l} for t,g in T for p,l in [qc(t)]]);display(QDF)
except Exception as e:print(e)
r=[]
if not GDF.empty:r.append({"model":"Gemini","accuracy":GDF.correct.mean(),"latency":GDF.lat.mean()})
if not QDF.empty:r.append({"model":"Qwen2.5-0.5B","accuracy":QDF.correct.mean(),"latency":QDF.lat.mean()})
CMP=pd.DataFrame(r)
if not CMP.empty:display(CMP.round(3))


# Research Note — Engineering Basis

This section is researched independently of the notebook results.

- **RAG:** Lewis et al. (2020) introduced retrieval-augmented generation to condition generation on retrieved non-parametric context. Here, retrieval supplies task guidance; A→B tests whether it helps. https://arxiv.org/abs/2005.11401
- **Dense retrieval:** Reimers & Gurevych (2019) showed Sentence-BERT enables efficient semantic similarity search, supporting MiniLM+FAISS as the first stage. https://arxiv.org/abs/1908.10084
- **Reranking:** Nogueira & Cho (2019) showed BERT reranking can improve query/passage ordering, supporting a dense-retrieval-then-cross-encoder design. https://arxiv.org/abs/1901.04085
- **LLM-as-judge:** Zheng et al. (2023) found LLM judges useful but documented position, verbosity, and self-enhancement biases. Judge scores are therefore supplemental to deterministic metrics. https://arxiv.org/abs/2306.05685
- **Local model:** Qwen2.5 provides small open models suitable for testing narrow local roles; this notebook tests 0.5B classification rather than assuming it can replace full synthesis. https://arxiv.org/abs/2412.15115

**Research-derived position:** retain retrieval/reranking and few-shot context only if ablation shows downstream value; do not let the LLM judge override deterministic evidence; use Qwen in the critical path only if measured accuracy is competitive.


# Automated Findings Memo — Measured Results/Conclusions


In [ ]:
def memo():
 if not RES or JD.empty or FAIL is None:return None
 j=chr(10).join(f"- **{i}:** {float(r['average']):.3f}" for i,r in JS.iterrows());q="comparison incomplete"
 if len(CMP)>=2:q=f"Qwen accuracy {CMP.iloc[1].accuracy:.3f} vs Gemini {CMP.iloc[0].accuracy:.3f}; latency {CMP.iloc[1].latency:.3f}s vs {CMP.iloc[0].latency:.3f}s."
 s=f'''## A/B/C metrics
{SUM.to_markdown()}

**Selected:** {BEST}

## Judge averages
{j}

**Failure:** {FAIL["case"]} — {FAIL["component"]}

**Mitigation decision F1:** {BEF} → {AFT}

**Gemini vs Qwen:** {q}'''
 display(Markdown(s));return s
FINDINGS=memo()


# Final Defended Conclusion


In [ ]:
def conclusion():
 if FINDINGS is None:return None
 a,b,c=[float(SUM.loc[x,"score"]) for x in "ABC"];q="Qwen comparison incomplete, so exclude it from the critical path."
 if len(CMP)>=2:
  g,qw=float(CMP.iloc[0].accuracy),float(CMP.iloc[1].accuracy);q=(f"Qwen accuracy {qw:.3f} is within 0.05 of Gemini {g:.3f}, so it is defensible only as an optional local classifier." if qw>=g-.05 else f"Qwen accuracy {qw:.3f} trails Gemini {g:.3f} by more than 0.05, so it is not justified in the critical path.")
 s=f'''## Defended architecture decision

Use **configuration {BEST}** because it achieved the highest held-out extraction score. Research makes retrieval, reranking, and judging plausible techniques; the notebook ablation decides whether they earn a place here: **A→B {a:.3f}→{b:.3f}; B→C {b:.3f}→{c:.3f}.**

Treat the 10-case LLM-as-judge pass as supporting evidence, not selection authority, because published work documents judge bias. Deterministic held-out metrics remain primary.

The required failure was **{FAIL["case"]}**, traced to **{FAIL["component"]}**. The mitigation changed mean decision F1 **{BEF:.3f}→{AFT:.3f}**; keep it only if that measured result improves or preserves performance.

For local inference, {q}

### Conclusion
**Ship the simplest A/B/C configuration that wins the held-out evaluation; keep deterministic validation and metrics around Gemini; retain RAG/reranking or dynamic few-shot context only when their measured ablations justify them; treat LLM-as-judge as supplemental; and use Qwen only for a narrow role if its measured accuracy supports the tradeoff.**

This conclusion is defended by both the cited research and the measured results.
'''
 display(Markdown(s));return s
DEFENDED_CONCLUSION=conclusion()


In [ ]:
p=[]
if not RES:p+=["A/B/C"]
if JD.empty or len(JD)!=10:p+=["judge"]
if FAIL is None:p+=["failure"]
if CMP.empty or len(CMP)<2:p+=["Qwen"]
if FINDINGS is None:p+=["findings"]
if DEFENDED_CONCLUSION is None:p+=["conclusion"]
print("READY FOR SUBMISSION" if not p else "NOT READY: "+", ".join(p))
print("HTML: jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb")
